# Multi-faceted Filtering

Layer metadata filters, a score threshold and diversity (MMR) on top of vector search.

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")
from common import *
import numpy as np

chunks, meta = [], []
for heading, body in split_sections(load_text()):
    for c in chunk_text(body, size=400, overlap=40):
        chunks.append(c)
        meta.append({"section": heading, "words": len(c.split())})
store = build_store(chunks, meta)
print(sorted({m["section"] for m in meta}))

### Facet 1: metadata filter
Restrict candidates by section (this is also how you enforce per-user access control).

In [ ]:
def search_filtered(query, sections=None, min_score=0.25, k=5):
    mask = np.array([sections is None or m["section"] in sections for m in meta])
    return [h for h in store.search(query, k * 3, mask=mask) if h["score"] >= min_score][:k]

query = "How do I reduce hallucinations?"
show(search_filtered(query, sections={"Hallucinations and Grounding", "Evaluating RAG Systems"}), width=100)

### Facet 2: diversity with MMR
Maximal Marginal Relevance picks results that are relevant **and** different from each other.

In [ ]:
def mmr(query, k=4, lam=0.6, pool=15):
    q = embed(query)[0]
    cand = [h["idx"] for h in store.search(query, pool)]
    chosen = []
    while cand and len(chosen) < k:
        def score(i):
            rel = store.vecs[i] @ q
            red = max((store.vecs[i] @ store.vecs[j] for j in chosen), default=0)
            return lam * rel - (1 - lam) * red
        best = max(cand, key=score)
        chosen.append(best); cand.remove(best)
    return [{"idx": i, "text": chunks[i], "score": float(store.vecs[i] @ q), "meta": meta[i]} for i in chosen]

hits = mmr("What affects RAG quality and speed?")
for h in hits: print(h["meta"]["section"], f"{h['score']:.2f}")
print(answer("What affects RAG quality and speed?", [h["text"] for h in hits]))